# A2 – Sequential vs Parallel Ensemble Benchmark

Compare the two ways `aifs-modal` can run an AIFS-ENS ensemble forecast:

- **Sequential** — `run_forecast(n_members=10, parallel_members=False)`: one GPU container runs all members back-to-back, reusing the loaded runner across members. Single `to_zarr` write at the end.
- **Parallel** — `run_forecast(n_members=10, parallel_members=True)`: one GPU container per member, each dumping a scratch zarr to a shared Modal volume; a CPU container then stacks the members along `ensemble_member` and writes once to icechunk.

Both produce the same output schema; only wall-clock time and Modal billing units differ.

In [ ]:
import datetime as dt
import os
import time

import icechunk
import xarray as xr

from aifs_modal import app, ingest_ifs_arraylake, run_forecast, settings

In [ ]:
n_members = 10
lead_time = 96  # hours

# init time used for both runs
# start date for IFS Brightband/Earthmover initial conditions: 1 day ago (safely within
# the rolling window)
date = dt.datetime.now(dt.UTC).replace(
    hour=0, minute=0, second=0, microsecond=0
) - dt.timedelta(days=1)

# checkpoint
checkpoint = settings.AIFS_ENS_CHECKPOINT

storage_bucket = "aifs-modal-unibe"

# IFS ArrayLake source repository
ifs_source_repo = "martibosch/ecmwf-ifs-hres-ics-open"

# write the two runs to distinct branches so they don't clobber each other
outputs_prefix = "aifs-outputs-bench-seq-vs-par"
outputs_branch_seq = "sequential"
outputs_branch_par = "parallel"

print(f"date       : {date}")
print(f"members    : {n_members}")
print(f"lead time  : {lead_time} h")

date       : 2026-05-09 00:00:00+00:00
members    : 10
lead time  : 96 h


## 1. Pre-ingest initial conditions

Both timing runs must start from already-ingested ICs so neither path pays
for the fetch. We call `ingest_ifs_arraylake` directly to ingest the ICs for
`date` and `date−6h` into the Modal IC Volume.

In [ ]:
start_date = (date - dt.timedelta(hours=6)).isoformat()
end_date = date.isoformat()

with app.run():
    ingest_ifs_arraylake.remote(
        start_date,
        end_date,
        source_repo=ifs_source_repo,
    )

## 2. Helper to open a forecast output

In [ ]:
outputs_storage = icechunk.tigris_storage(
    bucket=storage_bucket,
    prefix=outputs_prefix,
    region=os.environ["AWS_REGION"],
)
outputs_repo = icechunk.Repository.open_or_create(outputs_storage)
group = date.strftime("%Y-%m-%d/%Hz")


def open_forecast(branch):
    return xr.open_dataset(
        outputs_repo.readonly_session(branch).store,
        group=group,
        engine="zarr",
        zarr_format=3,
        chunks=None,
    )

  2026-05-10T13:22:02.144717Z  WARN aws_runtime::env_config::normalize: profile [plugins] ignored; sections in the AWS config file (other than [default]) must have a prefix i.e. [profile my-profile]
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-runtime-1.7.2/src/env_config/normalize.rs:121



## 3. Sequential run

`run_forecast` with `parallel_members=False` loops all members on a single GPU.
The runner and initial-conditions fields are loaded once and reused across all
members; only `runner.run(...)` is repeated per member.

In [ ]:
with app.run():
    _t0 = time.perf_counter()
    run_forecast.remote(
        date,
        storage_bucket,
        checkpoint=checkpoint,
        lead_time=lead_time,
        outputs_prefix=outputs_prefix,
        outputs_branch=outputs_branch_seq,
        n_members=n_members,
        parallel_members=False,
        keep_ics=True,
        overwrite=True,
    )
    seq_seconds = time.perf_counter() - _t0

print(f"sequential : {seq_seconds:.1f} s ({seq_seconds / n_members:.1f} s/member)")

sequential : 491.8 s (49.2 s/member)


In [ ]:
ds_seq = open_forecast(outputs_branch_seq)
ds_seq

<xarray.Dataset> Size: 15GB
Dimensions:          (ensemble_member: 10, init_time: 1, lead_time: 16,
                      lat: 721, lon: 1440, pressure: 13)
Coordinates:
  * ensemble_member  (ensemble_member) int64 80B 0 1 2 3 4 5 6 7 8 9
  * init_time        (init_time) datetime64[ns] 8B 2026-05-09
  * lead_time        (lead_time) timedelta64[us] 128B 0 days 06:00:00 ... 4 d...
  * lat              (lat) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * lon              (lon) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * pressure         (pressure) int64 104B 50 100 150 200 ... 700 850 925 1000
Data variables: (12/22)
    100u             (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    100v             (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    cp               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    hcc              (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    10u              (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    2t               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    ...               ...
    sf               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    tp               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    stl1             (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    tcw              (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    strd             (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    skt              (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...

## 4. Parallel run

`run_forecast` with `parallel_members=True` spawns one GPU container per
member. Members write their per-member output to a shared volume in parallel; a
CPU container consolidates them at the end.

In [ ]:
with app.run():
    _t0 = time.perf_counter()
    run_forecast.remote(
        date,
        storage_bucket,
        n_members=n_members,
        parallel_members=True,
        lead_time=lead_time,
        outputs_prefix=outputs_prefix,
        outputs_branch=outputs_branch_par,
        checkpoint=checkpoint,
        overwrite=True,
    )
    par_seconds = time.perf_counter() - _t0

print(f"parallel   : {par_seconds:.1f} s")

parallel   : 277.1 s


In [ ]:
ds_par = open_forecast(outputs_branch_par)
ds_par

<xarray.Dataset> Size: 15GB
Dimensions:          (ensemble_member: 10, init_time: 1, lead_time: 16,
                      lat: 721, lon: 1440, pressure: 13)
Coordinates:
  * ensemble_member  (ensemble_member) int64 80B 0 1 2 3 4 5 6 7 8 9
  * init_time        (init_time) datetime64[ns] 8B 2026-05-09
  * lead_time        (lead_time) timedelta64[us] 128B 0 days 06:00:00 ... 4 d...
  * lat              (lat) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * lon              (lon) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * pressure         (pressure) int64 104B 50 100 150 200 ... 700 850 925 1000
Data variables: (12/22)
    100u             (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    10v              (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    2t               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    cp               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    hcc              (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    2d               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    ...               ...
    strd             (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    sp               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    tcc              (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    tcw              (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    stl2             (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
    tp               (ensemble_member, init_time, lead_time, lat, lon) float32 664MB ...
Attributes:
    aifs_modal_status:  complete
    n_members:          10
    lead_time_hours:    96

## 5. Summary

In [ ]:
speedup = seq_seconds / par_seconds
print(f"sequential : {seq_seconds:7.1f} s")
print(f"parallel   : {par_seconds:7.1f} s")
print(f"speedup    : {speedup:7.2f}x ({n_members} members)")
print(f"sequential dims: {dict(ds_seq.sizes)}")
print(f"parallel   dims: {dict(ds_par.sizes)}")

sequential :   491.8 s
parallel   :   277.1 s
speedup    :    1.77x (10 members)
sequential dims: {'ensemble_member': 10, 'init_time': 1, 'lead_time': 16, 'lat': 721, 'lon': 1440, 'pressure': 13}
parallel   dims: {'ensemble_member': 10, 'init_time': 1, 'lead_time': 16, 'lat': 721, 'lon': 1440, 'pressure': 13}


## 6. Cleanup

Delete the outputs prefix so reruns of this notebook start from a clean slate.


In [ ]:
import storage_utils

storage_utils.delete_prefixes(
    storage_bucket,
    outputs_prefix,
)

deleted 12247 object(s) under s3://aifs-modal-unibe/aifs-outputs-bench-seq-vs-par
done — removed 12247 objects from s3://aifs-modal-unibe/
